# Day2 모델링 (5) - 최종 모델: 홀드아웃 확인, 제출 2, 변수별 영향력

05에서 정한 **고정된 레시피**를 쓴다. 이 노트북에서는 설정을 바꾸지 않는다.

- 레시피: LightGBM(튜닝) + XGBoost(튜닝) + Tanimoto SVM의 확률 평균, 임계값은 05의 OOF F1 곡선에서 평탄 구간의 중앙(5점 이동평균 정점)
- 1단계: 학습 80%로 학습해서 **봉인했던 홀드아웃 20%를 열어** 점수를 확인한다. 이 결과로 설정을 바꾸지 않는다. (홀드아웃은 처음에 OOF 최대점 0.38로 한 번 열었고, 그 구간이 평평하다는 점 때문에 임계값 규칙만 평탄 구간 중앙으로 바꿔 다시 계산했다. 모델과 하이퍼파라미터는 그대로다.)
- 2단계: 전체 train(100%)으로 다시 학습해 `predict_input.csv`를 예측하고 제출 2 파일을 만든다.
- 3단계: 변수별 영향력(SHAP)을 피처군별로 정리한다(교수님 평가 기준).

In [1]:
import json
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from xgboost import XGBClassifier

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from data import load_all  # noqa: E402
from models import TanimotoSVC  # noqa: E402
from preprocess import build_pipeline, build_selected_pipeline  # noqa: E402

warnings.filterwarnings("ignore")
SEED = 42
TAB = ROOT / "reports/tables"
CACHE = ROOT / "data/cache"
d = load_all(ROOT, seed=SEED, extra=True)
GROUPS = d.groups
ALL = ["cont", "rdkit", "fp", "maccs", "morgan_cnt", "ap_cnt", "tt_cnt"]
SVC_GROUPS = ["fp", "maccs", "morgan_cnt", "ap_cnt", "tt_cnt"]

best = json.loads((TAB / "best_params.json").read_text(encoding="utf-8"))
# 임계값 규칙: 05의 OOF F1 곡선은 0.32~0.50 구간이 거의 평평(최대에서 0.002 이내)해서 최대점 하나는 우연의 영향이 크다.
# 그래서 최대점 대신 5점 이동평균의 정점(평탄 구간 안쪽)을 쓴다. 홀드아웃은 이 규칙을 정하는 데 쓰지 않는다.
oof = pd.read_pickle(CACHE / "oof_05.pkl")
p_oof = np.mean(list(oof.values()), axis=0)
THS = np.round(np.arange(0.30, 0.71, 0.01), 2)
f1_oof = np.array([f1_score(d.y_tr.values, (p_oof >= t).astype(int)) for t in THS])
THRESH = float(THS[np.convolve(f1_oof, np.ones(5) / 5, mode="same").argmax()])
pd.DataFrame([{"규칙": "OOF F1의 5점 이동평균 정점", "임계값": THRESH, "OOF F1(임계값)": f1_oof[list(THS).index(THRESH)],
               "OOF 최대 F1": f1_oof.max(), "OOF 최대점 임계값": THS[f1_oof.argmax()],
               "OOF 예측 label 1 비율": (p_oof >= THRESH).mean()}]).round(4).to_csv(TAB / "final_threshold.csv", index=False)
print(f"임계값 {THRESH} (OOF 최대점은 {THS[f1_oof.argmax()]}, F1 {f1_oof.max():.4f} / 선택 임계값 F1 {f1_oof[list(THS).index(THRESH)]:.4f}) | SVC C: {best['svc_C']}")


def make_models():
    sel = lambda m: build_selected_pipeline(m, GROUPS, ALL, k=1000, seed=SEED)  # noqa: E731
    return {
        "LightGBM": sel(LGBMClassifier(**best["lightgbm"], subsample_freq=1, random_state=SEED, n_jobs=8, verbose=-1)),
        "XGBoost": sel(XGBClassifier(**best["xgboost"], tree_method="hist", n_jobs=8, random_state=SEED, eval_metric="logloss")),
        "TanimotoSVC": build_pipeline(TanimotoSVC(C=best["svc_C"]), GROUPS, SVC_GROUPS),
    }


def ensemble_proba(models, X):
    return np.mean([m.predict_proba(X)[:, 1] for m in models.values()], axis=0)

임계값 0.43 (OOF 최대점은 0.38, F1 0.8334 / 선택 임계값 F1 0.8328) | SVC C: 10


## 1. 학습 80%로 학습 → 홀드아웃 20% 확인 (처음이자 한 번)

In [2]:
models_tr = make_models()
for name, m in models_tr.items():
    m.fit(d.X_tr, d.y_tr)

y_ho = d.y_ho.values
proba_ho = {name: m.predict_proba(d.X_ho)[:, 1] for name, m in models_tr.items()}
proba_ho["앙상블(평균)"] = np.mean([proba_ho[n] for n in models_tr], axis=0)


def report(name, p):
    pred = (p >= THRESH).astype(int)
    return {"모델": name, "AUC": roc_auc_score(y_ho, p), f"F1@{THRESH}": f1_score(y_ho, pred), "F1@0.5": f1_score(y_ho, (p >= 0.5).astype(int)),
            "precision": precision_score(y_ho, pred), "recall": recall_score(y_ho, pred)}


rows = [report(n, p) for n, p in proba_ho.items()]
rows.append({"모델": "전부 1 예측(기준선)", "AUC": 0.5, f"F1@{THRESH}": f1_score(y_ho, np.ones_like(y_ho)), "F1@0.5": f1_score(y_ho, np.ones_like(y_ho)),
             "precision": y_ho.mean(), "recall": 1.0})
hold = pd.DataFrame(rows)
hold.round(4).to_csv(TAB / "final_holdout.csv", index=False)
print(hold.round(4).to_string(index=False))
tn, fp_, fn, tp = confusion_matrix(y_ho, (proba_ho["앙상블(평균)"] >= THRESH).astype(int)).ravel()
print(f"\n앙상블 혼동행렬(임계값 {THRESH}): TP {tp}, FP {fp_}(독성 있는데 없다고 한 것), FN {fn}, TN {tn}")

          모델    AUC  F1@0.43  F1@0.5  precision  recall
    LightGBM 0.8829   0.8269  0.8281     0.7859  0.8724
     XGBoost 0.8830   0.8230  0.8228     0.7833  0.8669
 TanimotoSVC 0.8615   0.8006  0.8026     0.7635  0.8416
     앙상블(평균) 0.8825   0.8200  0.8243     0.7798  0.8647
전부 1 예측(기준선) 0.5000   0.7049  0.7049     0.5443  1.0000

앙상블 혼동행렬(임계값 0.43): TP 786, FP 222(독성 있는데 없다고 한 것), FN 123, TN 539


## 2. 전체 train(100%)으로 재학습 → 제출 2 파일

홀드아웃 점수를 확인한 뒤 같은 레시피로 8,349건 전체를 학습한다. 설정은 바꾸지 않는다. 파일은 `submissions/predict_output_정태호_2.csv`에 저장한다(교수님 양식 `predict_output_<작성자>.csv`에 맞춰 제출할 때 파일명을 `predict_output_정태호.csv`로 바꿔 올린다).

In [3]:
models_all = make_models()
for name, m in models_all.items():
    m.fit(d.X, d.y)
proba_pred = ensemble_proba(models_all, d.X_pred)
pd.to_pickle(proba_pred, CACHE / "final_proba_pred.pkl")

sub = pd.DataFrame({"SMILES": d.smiles_pred, "output": (proba_pred >= THRESH).astype(int)})
out_path = ROOT / "submissions/predict_output_정태호_2.csv"
out_path.parent.mkdir(exist_ok=True)
sub.to_csv(out_path, index=False)
print(sub.shape, "| 예측 label 1 비율", round(sub["output"].mean(), 3), "(학습 데이터 0.544, 제출 1은 0.562)")

prev = ROOT / "submissions/predict_output_정태호.csv"
if prev.exists():
    p1 = pd.read_csv(prev)
    print("제출 1과 예측이 다른 행:", int((p1["output"].values != sub["output"].values).sum()), "/", len(sub))

(927, 2) | 예측 label 1 비율 0.595 (학습 데이터 0.544, 제출 1은 0.562)
제출 1과 예측이 다른 행: 63 / 927


## 3. 변수별 영향력 (SHAP)

LightGBM(학습 80%로 학습한 모델)의 홀드아웃 예측에 대해 SHAP 값을 계산한다. SHAP은 각 변수가 예측 확률(로그 오즈)을 얼마나 올리고 내렸는지의 크기다. **영향력이 크다는 것이 독성의 원인이라는 뜻은 아니다.**

In [4]:
import shap
from scipy.stats import spearmanr

lgb_pipe = models_tr["LightGBM"]
names = lgb_pipe.named_steps["prep"].get_feature_names_out()
mask = lgb_pipe.named_steps["sel"].get_support()
sel_names = [n.split("__", 1)[1] for n in names[mask]]
Xh = lgb_pipe[:-1].transform(d.X_ho)

sv = shap.TreeExplainer(lgb_pipe.named_steps["model"]).shap_values(Xh)
if isinstance(sv, list):
    sv = sv[1]
sv = np.asarray(sv)
if sv.ndim == 3:
    sv = sv[:, :, 1]


def group_of(col):
    for g in ("cont", "rdkit"):
        if col in GROUPS[g]:
            return g
    for g in ("maccs", "morgan_cnt", "ap_cnt", "tt_cnt"):
        if col.startswith(g + "_"):
            return g
    return "fp"


imp = pd.DataFrame({"변수": sel_names, "평균|SHAP|": np.abs(sv).mean(axis=0)})
imp["군"] = imp["변수"].map(group_of)
Xh_arr = np.asarray(Xh)
imp["방향"] = [("클수록 독성 없음(1) 쪽" if spearmanr(Xh_arr[:, j], sv[:, j])[0] > 0 else "클수록 독성 있음(0) 쪽") for j in range(Xh_arr.shape[1])]
imp = imp.sort_values("평균|SHAP|", ascending=False).reset_index(drop=True)
imp.head(30).round(4).to_csv(TAB / "final_shap_top30.csv", index=False)

grp = imp.groupby("군").agg(선택된_변수수=("변수", "size"), 합계_SHAP=("평균|SHAP|", "sum")).sort_values("합계_SHAP", ascending=False)
grp["영향력 비율(%)"] = (grp["합계_SHAP"] / grp["합계_SHAP"].sum() * 100).round(1)
grp.round(4).to_csv(TAB / "final_shap_groups.csv")
print(grp.round(4).to_string())
print()
print(imp.head(20).round(4).to_string(index=False))

            선택된_변수수  합계_SHAP  영향력 비율(%)
군                                      
ap_cnt          412   2.6862       34.0
rdkit           131   2.4606       31.1
fp              242   1.0613       13.4
morgan_cnt      110   0.6825        8.6
cont              4   0.5432        6.9
tt_cnt           73   0.3083        3.9
maccs            28   0.1582        2.0

                       변수  평균|SHAP|          군             방향
                    clogp    0.4960       cont 클수록 독성 있음(0) 쪽
                  rd_TPSA    0.2164      rdkit 클수록 독성 없음(1) 쪽
           morgan_cnt_807    0.1376 morgan_cnt 클수록 독성 없음(1) 쪽
            morgan_cnt_80    0.1050 morgan_cnt 클수록 독성 있음(0) 쪽
           rd_VSA_EState2    0.0823      rdkit 클수록 독성 없음(1) 쪽
                 ecfp_807    0.0818         fp 클수록 독성 없음(1) 쪽
              ap_cnt_1475    0.0760     ap_cnt 클수록 독성 있음(0) 쪽
               ap_cnt_836    0.0640     ap_cnt 클수록 독성 없음(1) 쪽
             rd_PEOE_VSA6    0.0611      rdkit 클수록 독성 있음(0) 쪽
           rd_EStat